# This notebook is to sort by channel and then cluster each channel

**USE Cleaned Data!**

In [ ]:
from treeview_gen import df_to_treeview
import pandas as pd
import os
import re
import shutil
from time import sleep

regex_df = pd.read_csv("/home/derfelt/PartchLabFiles/Circadian Rhythms/Pilot Study/cleaned_data/feature_reduction_cleaned_renamed/HistDiff_results_24h_bmal1ref_REDONE_WELL-DROPPED_FeatRed.csv", index_col=0, nrows=1)
pattern = r'Region (.+?) \(global\)'
channels_found = set()
for col in regex_df.columns:
    found = re.search(pattern, col)
    if found:
        channels_found.add(found.group(1))

print(channels_found)


In [ ]:
def gen_treeview_channel(df: pd.DataFrame, channel_name: str, out_dir: str, sortIndex: bool = True) -> list[str]:
    """Generates a treeviewe file set for a given channel

    Args:
        df (pd.DataFrame): channel data frame
        channel_name (str): the name of the channel
        out_dir (str): the file path where the file will be outputted to (this creates a folder)
    """
    
    out_dir_channel = os.path.join(out_dir, channel_name)
    
    if os.path.exists(out_dir_channel):
        shutil.rmtree(out_dir_channel)
        sleep(0.5)
    os.makedirs(out_dir_channel, exist_ok=True)
    
    out_dir_channel = os.path.join(out_dir_channel, f"{channel_name.replace("/","_")}_treeview")
    
    df_to_treeview(df=df, outFile=out_dir_channel, sortIndex=sortIndex, row_cluster=False)
    
    cdt_path = f"{out_dir_channel}.cdt"
    return read_tv_col_order(cdt_path)


def read_tv_col_order(file_path: str) -> list[str]:
    col_df = pd.read_csv(file_path, index_col=[0,1,2], sep="\t")
    return col_df.columns.tolist()

In [ ]:
test_cdt = "/home/derfelt/PartchLabFiles/Circadian Rhythms/Pilot Study/Heatmaps (treeview)/channel sorted/HistDiff_results_24h_bmal1ref_FeatRed_WELL-DROPPED/Alexa 488 shortened/Alexa 488 shortened_treeview.cdt"
cdt_df = pd.read_csv(test_cdt, sep="\t", index_col=[0,1,2])

print(cdt_df.columns.tolist())

## Final Loop

In [ ]:
cleaned_file_dir = "/home/derfelt/PartchLabFiles/Circadian Rhythms/Pilot Study/cleaned_data/feature_reduction_cleaned"
out_dir = "/home/derfelt/PartchLabFiles/Circadian Rhythms/Pilot Study/Heatmaps (treeview)/channel sorted"

for file in os.listdir(cleaned_file_dir):
    curr_file = os.path.join(cleaned_file_dir, file)
    print(curr_file)
    raw_name = os.path.splitext(file)[0]
    print(raw_name)
    
    raw_out_dir = os.path.join(out_dir, raw_name)
    print(raw_out_dir)
    
    # make a folder
    if os.path.exists(raw_out_dir):
        shutil.rmtree(raw_out_dir)
        sleep(0.5)
    os.makedirs(raw_out_dir)
    
    
    curr_df = pd.read_csv(curr_file, index_col=0)
    
    # grab channels from data
    channel_df = {ch: curr_df[[col for col in curr_df.columns if ch in col]] for ch in channels_found}
    
    # grab the column orders and export out the channel clustering
    col_order = []
    for name, df in channel_df.items():
        order = gen_treeview_channel(df, name.replace("/", "_"), raw_out_dir)
        col_order.extend(order)
    
    print(col_order)
    
    reorder_df = curr_df[col_order]
    
    final_out_path = os.path.join(raw_out_dir, f"{raw_name}_SORTED-CLUSTERED_channels")
    df_to_treeview(df=reorder_df, outFile=final_out_path, sortIndex=True, row_cluster=False, col_cluster=False)

# We made an error:

- **ALL HEATMAPS MUST HAVE THE SAME COLUMN ORDERINGS AND CLUSTERING**
- This means we need to do the following:
1. Combine all the datasets together
2. Split channels
3. cluster columns of combined but channel split dataset
4. retrieve ordering
5. sort ids and apply ordering

In [ ]:
cleaned_file_dir = "/home/derfelt/PartchLabFiles/Circadian Rhythms/Pilot Study/cleaned_data/feature_reduction_cleaned_renamed"
out_dir = "/home/derfelt/PartchLabFiles/Circadian Rhythms/Pilot Study/Heatmaps (treeview)/channel sorted"

In [ ]:
# lets remove the orginal channel sorted folder
def clean_folder(path: str) -> None:
    if os.path.exists(path):
        shutil.rmtree(path)
        sleep(0.5)
    os.makedirs(path, exist_ok=True)

In [ ]:
clean_folder(out_dir)

In [ ]:
# combine data sets
renamed_datasets = "/home/derfelt/PartchLabFiles/Circadian Rhythms/Pilot Study/cleaned_data/feature_reduction_cleaned_renamed"

dfs = []
for file in os.listdir(renamed_datasets):
    curr_file = os.path.join(renamed_datasets, file)
    curr_df = pd.read_csv(curr_file, index_col=0)
    dfs.append(curr_df)

concat_df = pd.concat(dfs)
display(concat_df)
print(concat_df.shape)

concat_df.to_csv('/home/derfelt/PartchLabFiles/Circadian Rhythms/Pilot Study/Heatmaps (treeview)/channel sorted/combined_data.csv')

In [ ]:
# sort, cluster, and grab/create final row ordering

channel_df = {ch: concat_df[[col for col in concat_df.columns if ch in col]] for ch in channels_found}

channel_out_dir = "/home/derfelt/PartchLabFiles/Circadian Rhythms/Pilot Study/Heatmaps (treeview)/channel sorted/"

# if os.path.exists(channel_out_dir):
#     shutil.rmtree(channel_out_dir)
#     sleep(0.5)
# os.makedirs(channel_out_dir, exist_ok=True)

final_order = list()
for name, df in channel_df.items():
    print(name)
    print(f"Shape: {df.shape}\n")
    
    fix_name = name.replace("/", "_")
    # curr_channel_out = os.path.join(channel_out_dir, fix_name)
    # clean_folder(curr_channel_out)
    
    # file_out = os.path.join(curr_channel_out, f"{fix_name}_CHANNEL-SORTED")
    order = gen_treeview_channel(df=df, channel_name=fix_name, out_dir=channel_out_dir, sortIndex=False)
    final_order.extend(order)

    
print(len(final_order))
print(final_order)    
    
    

## Final loop

In [ ]:
out_dir = "/home/derfelt/PartchLabFiles/Circadian Rhythms/Pilot Study/Heatmaps (treeview)/channel sorted/combined_dataset"
concat_df = concat_df[final_order]
display(concat_df)
df_to_treeview(df=concat_df, outFile=out_dir, row_cluster=False, col_cluster=False, sortIndex=True)

In [ ]:
out_dir = "/home/derfelt/PartchLabFiles/Circadian Rhythms/Pilot Study/Heatmaps (treeview)/channel sorted/feat_red_channel_sorted/"
clean_folder(out_dir)
for file in os.listdir(cleaned_file_dir):
    curr_file = os.path.join(cleaned_file_dir, file)
    
    curr_df = pd.read_csv(curr_file, index_col=0)
    curr_df = curr_df[final_order]
    out_file = os.path.splitext(file)[0]
    
    df_to_treeview(df=curr_df, outFile=os.path.join(out_dir, out_file), row_cluster=False, col_cluster=False, sortIndex=True)
    

# Split by bmal1 and per2 

## We need the following:

1. all plates/timepoints and treatment in one heatmap (but separate heatmap for Bmal1 and separate heatmap for Per2) - this one is probably the one you want to use for clustering features within channel and using that order in all other heatmaps
2. 'no treatment' heatmap (one for Bmal1 and one for Per2) - only no treatment rows but all timepoints in one heatmap 
3. 'Dex heatmap (one for Bmal1 and one for Per2) - only Dex rows but all timepoints in one heatmap 
4. 'PF' heatmap (one for Bmal1 and one for Per2) - only PF rows but all timepoints in one heatmap 
5. '24h' heatmap (one for Bmal1 and one for Per2) - only '24h' plate but all treatments in one heatmap 
6. '66h' heatmap (one for Bmal1 and one for Per2) - only '66h' plate but all treatments in one heatmap 
7. '72h' heatmap (one for Bmal1 and one for Per2) - only '72h' plate but all treatments in one heatmap

bmal1

In [56]:
bmal_out_dir = "/home/derfelt/PartchLabFiles/Circadian Rhythms/Pilot Study/Heatmaps (treeview)/channel sorted/bmal1"

curr_df = pd.read_csv('/home/derfelt/PartchLabFiles/Circadian Rhythms/Pilot Study/cleaned_data/feature_reduction_cleaned_renamed/HistDiff_results_24h_bmal1ref_REDONE_WELL-DROPPED_FeatRed.csv', index_col=0)
display(curr_df)


,Nuclei - Cell Region Alexa 488 shortened (global) Profile 2/5 SER-Hole,Nuclei - Ring Region HOECHST 33342 expanded (global) Radial Relative Deviation SER-Ridge,Nuclei - Ring Region SYTO14 Rabbitts (global) Profile 2/5 SER-Hole,Nuclei - Cell Region TMR/AF555 Rabbitts (global) Profile 3/5 SER-Bright,Nuclei - Nucleus Region SYTO14 Rabbitts (global) Radial Mean Ratio SER-Valley,Nuclei - Intensity Membrane Region MitoTracker Deep Red (global) Contrast,Nuclei - Cytoplasm Region MitoTracker Deep Red (global) Threshold Compactness 30%,Nuclei - Cell Region MitoTracker Deep Red (global) Profile 2/5,Nuclei - Nucleus Region HOECHST 33342 expanded (global) Symmetry 13 SER-Bright,Nuclei - Ring Region Alexa 488 shortened (global) Profile 1/5,...,Nuclei - Nucleus Region HOECHST 33342 expanded (global) Symmetry 05 SER-Bright,Nuclei - Membrane Region HOECHST 33342 expanded (global) Profile 1/5,Nuclei - Cytoplasm Region MitoTracker Deep Red (global) Threshold Compactness 40%,Nuclei - Membrane Region MitoTracker Deep Red (global) Threshold Compactness 30% SER-Edge,Nuclei - Ring Region HOECHST 33342 expanded (global) Gabor Min 2 px w2,Nuclei - Ring Region MitoTracker Deep Red (global) Radial Mean Ratio SER-Edge,Nuclei - Nucleus Region TMR/AF555 Rabbitts (global) Threshold Compactness 30%,Nuclei - Cell Region HOECHST 33342 expanded (global) Threshold Compactness 30%,Nuclei - Intensity Ring Region HOECHST 33342 expanded (global) StdDev,Nuclei - Membrane Region HOECHST 33342 expanded (global) Profile 2/5 SER-Hole
human_readable_ids,,,,,,,,,,,,,,,,,,,,,
NO-TREATMENT._.A10._.24h._.bmal1ref,0.002570,-0.000127,0.000008,-0.000064,0.000220,-0.000272,-0.000084,-0.000118,-0.000523,0.000083,...,-0.000169,0.000184,0.000054,-0.000040,1.205608e-04,-0.000057,-0.000042,-0.000877,0.003708,-1.190946e-05
NO-TREATMENT._.A11._.24h._.bmal1ref,0.000060,0.000244,-0.000007,-0.000014,-0.000093,-0.001290,0.000673,0.000098,-0.000067,-0.000018,...,-0.000095,0.000099,0.000093,-0.000161,-5.480439e-06,0.000100,0.000214,0.000330,0.002808,-8.364908e-06
NO-TREATMENT._.A12._.24h._.bmal1ref,-0.001167,-0.000180,-0.000007,-0.001610,-0.000058,0.001164,0.000586,0.000199,-0.000086,0.000102,...,-0.000194,-0.000259,-0.000050,-0.000040,-4.587930e-04,0.000036,0.000293,-0.000130,0.001278,-9.000905e-07
NO-TREATMENT._.A13._.24h._.bmal1ref,0.000782,0.000297,-0.000051,-0.000051,0.000504,-0.001473,-0.000314,-0.000191,0.000255,-0.000177,...,0.000504,0.000427,-0.000115,0.000018,-3.523404e-07,0.000231,-0.000153,0.001274,0.001728,2.906337e-06
NO-TREATMENT._.A14._.24h._.bmal1ref,-0.008598,-0.000412,0.000001,-0.000047,-0.000597,0.019484,-0.000103,0.001361,0.001008,0.002145,...,-0.000221,-0.000301,0.000267,0.001576,2.909721e-05,-0.001983,0.000151,-0.002635,0.001123,-9.838245e-07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
PF_per2._.O05._.24h._.bmal1ref,0.000008,0.000156,-0.000243,0.060838,0.017811,0.012321,0.000696,0.000176,-0.000121,-0.001537,...,-0.001347,-0.000475,-0.000682,-0.007427,8.333097e-05,-0.002442,-0.000373,-0.002786,-0.103613,-2.442106e-05
PF_per2._.O06._.24h._.bmal1ref,0.001958,0.000775,-0.000199,0.053993,0.013330,0.018570,-0.000036,0.000676,-0.000229,-0.001636,...,-0.005142,-0.000199,-0.000339,-0.013295,2.586701e-04,-0.000923,-0.000111,-0.002231,-0.133188,-4.272012e-05
PF_per2._.O07._.24h._.bmal1ref,0.000092,0.000687,-0.000209,0.071508,0.023982,0.018314,0.000377,0.000730,-0.000370,-0.000132,...,-0.003594,-0.000619,0.000531,-0.007786,2.129217e-03,-0.001837,-0.000274,0.001534,-0.098908,-5.196712e-05
